In [7]:
# ============================================================
# 📌 CELL 1 — IMPORT & PATH CONFIGURATION
# Deskripsi:
# - Mengimpor semua library yang dibutuhkan
# - Menentukan path STEAD, K-NET, dan output multi-domain
# ============================================================

import numpy as np
import pandas as pd
import h5py
import obspy
import scipy.signal as sig
from scipy.signal.windows import tukey
from pathlib import Path
from tqdm import tqdm  # progress bar

# === PATH STEAD ===
STEAD_BASE = Path("/Volumes/Extreme SSD/stream_stead/data_stead")
STEAD_CSV  = STEAD_BASE / "merge.csv"
STEAD_H5   = STEAD_BASE / "merge.hdf5"

# === PATH K-NET ===
KNET_ROOT  = Path("/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net")
KNET_OUT   = Path("/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net_ac3")
KNET_OUT.mkdir(exist_ok=True)

# === OUTPUT MULTI-DOMAIN ===
MULTI_OUT  = STEAD_BASE


In [8]:
# ============================================================
# 📌 CELL 2 — PREPROCESSING FUNCTION
# Deskripsi:
# - Membersihkan sinyal (detrend)
# - Mengurangi artefak ujung (taper Tukey)
# - Menyaring frekuensi penting (bandpass 1–20 Hz)
# - Menormalisasi per-trace agar stabil untuk AC-3
# - Menghasilkan output (6000, 3) float32
# ============================================================

def preprocess_trace(x, bandpass=True):
    x = x.astype(np.float32)

    # detrend
    x = sig.detrend(x, axis=0, type="linear")

    # taper
    win = tukey(x.shape[0], alpha=0.05)
    x = x * win[:, None]

    # bandpass
    if bandpass:
        fs = 100.0
        sos = sig.butter(4, [1.0, 20.0], btype="bandpass", fs=fs, output="sos")
        x = sig.sosfiltfilt(sos, x, axis=0)

    # normalisasi robust
    std = np.std(x)
    if std > 0:
        x = x / (std + 1e-6)

    # clipping
    x = np.clip(x, -5.0, 5.0)
    return x


In [9]:
# ============================================================
# 📌 CELL 3 — STEAD LOADER & BUILDER AC-3
# Deskripsi:
# - Memuat metadata STEAD (merge.csv)
# - Memilih event near-field dan noise
# - Mengambil waveform dari HDF5
# - Preprocessing menggunakan fungsi AC-3
# - Menampilkan progress bar untuk setiap trace
# ============================================================

def load_stead_waveform(h5_file, trace_name):
    ds = h5_file.get("data/" + str(trace_name))
    if ds is None:
        return None
    data = np.array(ds)
    if data.shape != (6000, 3):
        return None
    return data

def build_stead_ac3(csv_path, h5_path, max_ev=20000, max_ns=20000):
    df = pd.read_csv(csv_path)
    print("Total STEAD rows:", len(df))

    df_ev = df[
        (df.trace_category == "earthquake_local") &
        (df.source_distance_km <= 20) &
        (df.source_magnitude >= 3)
    ].copy()

    df_ns = df[df.trace_category == "noise"].copy()

    df_ev = df_ev.sample(min(max_ev, len(df_ev)), random_state=42)
    df_ns = df_ns.sample(min(max_ns, len(df_ns)), random_state=42)

    print("STEAD event:", len(df_ev), "noise:", len(df_ns))

    def build_Xy(df_subset, label):
        X_list, y_list = [], []
        with h5py.File(h5_path, "r") as f:
            for tr_name in tqdm(df_subset["trace_name"].values, desc=f"STEAD label={label}"):
                raw = load_stead_waveform(f, tr_name)
                if raw is None:
                    continue
                x_proc = preprocess_trace(raw)
                X_list.append(x_proc)
                y_list.append(label)
        X = np.stack(X_list, axis=0)
        y = np.array(y_list, dtype=np.int64)
        return X, y

    X_ev, y_ev = build_Xy(df_ev, 1)
    X_ns, y_ns = build_Xy(df_ns, 0)

    X = np.concatenate([X_ev, X_ns], axis=0)
    y = np.concatenate([y_ev, y_ns], axis=0)
    return X, y


In [10]:
# ============================================================
# 📌 CELL 4 — K-NET LOADER & BUILDER AC-3
# Deskripsi:
# - Menelusuri semua file K-NET secara rekursif
# - Memuat waveform 3 komponen
# - Resample ke 100 Hz, pad/truncate ke 6000 sampel
# - Preprocessing identik dengan STEAD
# - Menampilkan progress bar
# ============================================================

def list_knet_files(root):
    root = Path(root)
    exts = ["*.mseed", "*.txt", "*.dat"]
    files = []
    for ext in exts:
        files.extend(root.rglob(ext))
    return files

def load_knet_waveform(path):
    try:
        st = obspy.read(path)
    except:
        return None

    if len(st) < 3:
        return None

    st.sort(keys=["channel"])

    for tr in st:
        if tr.stats.sampling_rate != 100:
            tr.resample(100)

    data = []
    for tr in st[:3]:
        x = tr.data.astype(np.float32)
        if len(x) >= 6000:
            x = x[:6000]
        else:
            pad = np.zeros(6000 - len(x), dtype=np.float32)
            x = np.concatenate([x, pad])
        data.append(x)

    return np.stack(data, axis=1)

def build_knet_ac3(root, label=1):
    files = list_knet_files(root)
    print("Total K-NET files:", len(files))

    X_list, y_list = [], []
    for f in tqdm(files, desc="K-NET processing"):
        raw = load_knet_waveform(f)
        if raw is None:
            continue
        x_proc = preprocess_trace(raw)
        X_list.append(x_proc)
        y_list.append(label)

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    return X, y


In [11]:
# ============================================================
# 📌 CELL 5 — RUN STEAD → AC-3
# Deskripsi:
# - Menjalankan builder STEAD
# - Menyimpan X_stead_ac3.npy dan y_stead_ac3.npy
# ============================================================

X_stead, y_stead = build_stead_ac3(STEAD_CSV, STEAD_H5)
np.save(STEAD_BASE / "X_stead_ac3.npy", X_stead)
np.save(STEAD_BASE / "y_stead_ac3.npy", y_stead)

print("Saved STEAD AC-3:", X_stead.shape, y_stead.shape)


/var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/ipykernel_47919/519300741.py:21: DtypeWarning: Columns (7,11,13,14,24,25,26,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Total STEAD rows: 1265657
STEAD event: 2515 noise: 20000


STEAD label=0: 100%|██████████| 20000/20000 [02:45<00:00, 121.00it/s]


Saved STEAD AC-3: (22515, 6000, 3) (22515,)


In [13]:
# ============================================================
# 📌 DIAGNOSTIC CELL — CEK FILE K-NET
# Deskripsi:
# - Menampilkan 20 file pertama yang ditemukan
# - Memastikan ekstensi file benar
# - Memastikan struktur folder terbaca
# ============================================================

from pathlib import Path

KNET_ROOT = Path("/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net")

files = list(KNET_ROOT.rglob("*"))
print("Total entries:", len(files))

for f in files[:20]:
    print(f)


Total entries: 8621
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20260115071300.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20221002000200.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20191219152100.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20201212161900.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20211129214000.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/.DS_Store
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20241126224700.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20250226145300.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20161228213800.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20231120060100.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20170105025300.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20150713025200.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20181128112300.knt
/Volumes/Local Disk/Code_Git/S3_code/seismic/k_net/20260115071300.knt/HKD125260

In [14]:
# ============================================================
# 📌 CELL — LOADER K-NET (FORMAT .EW / .NS / .UD)
# Deskripsi:
# - Menelusuri setiap folder *.knt
# - Mendeteksi file komponen: .EW, .NS, .UD
# - Membaca masing-masing komponen sebagai 1D waveform
# - Menggabungkan menjadi array (6000, 3)
# - Resample ke 100 Hz
# - Pad/truncate ke 6000 sampel
# - Mengabaikan file non-waveform (.ps.gz)
# ============================================================

def load_knet_folder(folder):
    folder = Path(folder)

    # cari file komponen
    ew = list(folder.glob("*.EW"))
    ns = list(folder.glob("*.NS"))
    ud = list(folder.glob("*.UD"))

    if not (ew and ns and ud):
        return None  # tidak lengkap 3 komponen

    # baca waveform
    try:
        tr_ew = obspy.read(str(ew[0]))[0]
        tr_ns = obspy.read(str(ns[0]))[0]
        tr_ud = obspy.read(str(ud[0]))[0]
    except:
        return None

    # resample ke 100 Hz
    for tr in [tr_ew, tr_ns, tr_ud]:
        if tr.stats.sampling_rate != 100:
            tr.resample(100)

    # pad/truncate ke 6000 sampel
    def fix_len(x):
        x = x.astype(np.float32)
        if len(x) >= 6000:
            return x[:6000]
        pad = np.zeros(6000 - len(x), dtype=np.float32)
        return np.concatenate([x, pad])

    ew = fix_len(tr_ew.data)
    ns = fix_len(tr_ns.data)
    ud = fix_len(tr_ud.data)

    # gabungkan jadi (6000, 3)
    return np.stack([ew, ns, ud], axis=1)


In [15]:
# ============================================================
# 📌 CELL — BUILDER K-NET → AC-3
# Deskripsi:
# - Menelusuri semua folder *.knt
# - Memuat waveform 3 komponen dari setiap folder
# - Preprocessing menggunakan fungsi AC-3
# - Menampilkan progress bar
# - Menghasilkan X_knet_ac3.npy dan y_knet_ac3.npy
# ============================================================

def build_knet_ac3(root, label=1):
    root = Path(root)
    folders = [f for f in root.iterdir() if f.is_dir() and f.suffix == ".knt"]

    print("Total K-NET event folders:", len(folders))

    X_list, y_list = [], []

    for folder in tqdm(folders, desc="K-NET processing"):
        raw = load_knet_folder(folder)
        if raw is None:
            continue

        x_proc = preprocess_trace(raw)
        X_list.append(x_proc)
        y_list.append(label)

    if len(X_list) == 0:
        raise ValueError("Tidak ada waveform K-NET valid yang ditemukan.")

    X = np.stack(X_list, axis=0)
    y = np.array(y_list, dtype=np.int64)
    return X, y


In [16]:
# ============================================================
# 📌 CELL — RUN K-NET → AC-3
# Deskripsi:
# - Menjalankan builder K-NET
# - Menyimpan X_knet_ac3.npy dan y_knet_ac3.npy
# ============================================================

X_knet, y_knet = build_knet_ac3(KNET_ROOT, label=1)

np.save(KNET_OUT / "X_knet_ac3.npy", X_knet)
np.save(KNET_OUT / "y_knet_ac3.npy", y_knet)

print("Saved K-NET AC-3:", X_knet.shape, y_knet.shape)


Total K-NET event folders: 12


K-NET processing: 100%|██████████| 12/12 [00:00<00:00, 44.23it/s]

Saved K-NET AC-3: (12, 6000, 3) (12,)


In [17]:
# ============================================================
# 📌 CELL — BUILDER MULTI-DOMAIN (STEAD + K-NET)
# Deskripsi:
# - Memuat dataset STEAD AC-3 dan K-NET AC-3
# - Membuat domain label (0=STEAD, 1=K-NET)
# - Menggabungkan seluruh data menjadi dataset multi-domain
# - Menyimpan X_multi_ac3.npy, y_multi_ac3.npy, d_multi_ac3.npy
# ============================================================

from tqdm import tqdm

# Load STEAD
print("Loading STEAD...")
X_stead = np.load(STEAD_BASE / "X_stead_ac3.npy")
y_stead = np.load(STEAD_BASE / "y_stead_ac3.npy")
d_stead = np.zeros_like(y_stead)

# Load K-NET
print("Loading K-NET...")
X_knet = np.load(KNET_OUT / "X_knet_ac3.npy")
y_knet = np.load(KNET_OUT / "y_knet_ac3.npy")
d_knet = np.ones_like(y_knet)

# Gabungkan multi-domain
print("Merging datasets...")
X_multi = np.concatenate([X_stead, X_knet], axis=0)
y_multi = np.concatenate([y_stead, y_knet], axis=0)
d_multi = np.concatenate([d_stead, d_knet], axis=0)

# Simpan
np.save(MULTI_OUT / "X_multi_ac3.npy", X_multi)
np.save(MULTI_OUT / "y_multi_ac3.npy", y_multi)
np.save(MULTI_OUT / "d_multi_ac3.npy", d_multi)

print("Saved multi-domain dataset:")
print("X:", X_multi.shape)
print("y:", y_multi.shape)
print("d:", d_multi.shape)


Loading STEAD...
Loading K-NET...
Merging datasets...
Saved multi-domain dataset:
X: (22527, 6000, 3)
y: (22527,)
d: (22527,)


In [19]:
# ============================================================
# 📌 CELL — LOAD MULTI-DOMAIN DATASET
# Deskripsi:
# - Memuat X_multi_ac3.npy, y_multi_ac3.npy, d_multi_ac3.npy
# - Menampilkan progress bar saat loading
# - Menampilkan bentuk dataset untuk verifikasi
# ============================================================

from tqdm import tqdm

print("Loading multi-domain dataset...")

X_multi = np.load(MULTI_OUT / "X_multi_ac3.npy")
y_multi = np.load(MULTI_OUT / "y_multi_ac3.npy")
d_multi = np.load(MULTI_OUT / "d_multi_ac3.npy")

print("Dataset loaded successfully:")
print("X_multi:", X_multi.shape)
print("y_multi:", y_multi.shape)
print("d_multi:", d_multi.shape)


Loading multi-domain dataset...
Dataset loaded successfully:
X_multi: (22527, 6000, 3)
y_multi: (22527,)
d_multi: (22527,)


In [20]:
# ============================================================
# 📌 CELL 1 — LOAD MULTI-DOMAIN DATASET
# Deskripsi:
# - Memuat X_multi_ac3.npy, y_multi_ac3.npy, d_multi_ac3.npy
# - Menampilkan bentuk dataset untuk verifikasi
# ============================================================

from tqdm import tqdm

print("Loading multi-domain dataset...")

X = np.load(MULTI_OUT / "X_multi_ac3.npy")
y = np.load(MULTI_OUT / "y_multi_ac3.npy")
d = np.load(MULTI_OUT / "d_multi_ac3.npy")

print("Dataset loaded successfully:")
print("X:", X.shape)
print("y:", y.shape)
print("d:", d.shape)


Loading multi-domain dataset...
Dataset loaded successfully:
X: (22527, 6000, 3)
y: (22527,)
d: (22527,)


In [21]:
# ============================================================
# 📌 CELL 2 — TRAIN/VALIDATION SPLIT
# Deskripsi:
# - Membagi dataset menjadi train dan validation
# - Menggunakan stratify berdasarkan label y
# ============================================================

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, y_train.shape)
print("Val:", X_val.shape, y_val.shape)


Train: (18021, 6000, 3) (18021,)
Val: (4506, 6000, 3) (4506,)


In [22]:
# ============================================================
# 📌 CELL 3 — BUILD MODEL AC-3
# Deskripsi:
# - Membangun arsitektur CNN 1D untuk input (6000, 3)
# - Output binary classification (event/noise)
# - Optimizer Adam, loss binary crossentropy
# ============================================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_ac3_model():
    inputs = keras.Input(shape=(6000, 3))

    x = layers.Conv1D(32, 7, activation="relu", padding="same")(inputs)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(64, 5, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, activation="relu", padding="same")(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_ac3_model()
model.summary()


2026-03-08 11:18:33.777252: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Pro
2026-03-08 11:18:33.777303: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 18.00 GB
2026-03-08 11:18:33.777310: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 6.66 GB
2026-03-08 11:18:33.777347: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-08 11:18:33.777358: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 6000, 3)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 6000, 32)       │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 3000, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 3000, 64)       │        10,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 1500, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 1500, 128)      │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 750, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 750, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 183,681 (717.50 KB)

 Trainable params: 183,681 (717.50 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
# ============================================================
# 📌 CELL 4 — TRAINING AC-3 (RESUME-SAFE)
# Deskripsi:
# - Menjalankan training dengan checkpoint
# - Menyimpan model terbaik berdasarkan val_loss
# - Training dapat dilanjutkan kapan saja
# ============================================================

ckpt_path = MULTI_OUT / "ac3_checkpoint.keras"

checkpoint = keras.callbacks.ModelCheckpoint(
    filepath=str(ckpt_path),
    save_best_only=True,
    monitor="val_loss",
    mode="min"
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[checkpoint],
    verbose=1  # progress bar
)

print("Training selesai. Model terbaik disimpan di:", ckpt_path)


Epoch 1/20


2026-03-08 11:18:45.233618: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


564/564 ━━━━━━━━━━━━━━━━━━━━ 19s 31ms/step - accuracy: 0.9017 - loss: 0.1810 - val_accuracy: 0.9734 - val_loss: 0.0783
Epoch 2/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9779 - loss: 0.0789 - val_accuracy: 0.9834 - val_loss: 0.0703
Epoch 3/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9787 - loss: 0.0807 - val_accuracy: 0.9814 - val_loss: 0.0734
Epoch 4/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9127 - loss: 4.2131 - val_accuracy: 0.8879 - val_loss: 7.1866
Epoch 5/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.8519 - loss: 4.8102 - val_accuracy: 0.8879 - val_loss: 1.6815
Epoch 6/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.8614 - loss: 1.4593 - val_accuracy: 0.8879 - val_loss: 0.6882
Epoch 7/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.8664 - loss: 0.6463 - val_accuracy: 0.8953 - val_loss: 0.2642
Epoch 8/20
564/564 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.8930 - loss: 0.3221 - val_accurac